In [43]:
import os
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from collections import Counter
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform

In [44]:
# GLOBAL PARAMETERS

IMG_SIZE     = 64
N_COMPONENTS = 150
RANDOM_STATE = 42

LABELED_RATIO   = 0.20
UNLABELED_RATIO = 0.70
TEST_RATIO      = 0.10

DATASET_DIR = "bloodcells_dataset"

In [45]:
# READING FROM FOLDERS

def load_images(folder, has_labels=True):
    X, y, label_map = [], [], {}
    uploaded = 0

    if has_labels:
        classi = sorted([c for c in os.listdir(folder)
                         if os.path.isdir(os.path.join(folder, c))])
        
        for idx, classe in enumerate(classi):
            label_map[classe] = idx
            class_folder = os.path.join(folder, classe)
            
            for filename in os.listdir(class_folder):
                if not filename.lower().endswith(('.jpg', '.png', '.jpeg')):
                    continue
                    
                path = os.path.join(class_folder, filename)
                img  = Image.open(path).convert('L')
                img  = img.resize((IMG_SIZE, IMG_SIZE))
                
                vettore = np.array(img).flatten() / 255.0
                X.append(vettore)
                y.append(idx)
                uploaded += 1
    else:
        for filename in os.listdir(folder):
            if not filename.lower().endswith(('.jpg', '.png', '.jpeg')):
                continue
            path = os.path.join(folder, filename)
            img  = Image.open(path).convert('L')
            img  = img.resize((IMG_SIZE, IMG_SIZE))
            vettore = np.array(img).flatten() / 255.0
            X.append(vettore)
            uploaded += 1

    print(f"Total images found: {uploaded}")

    if has_labels:
        return np.array(X), np.array(y), label_map
    else:
        return np.array(X), None, None

In [46]:
# INFO AND UPLOAD

X_all, y_all, label_map = load_images(DATASET_DIR, has_labels=True)

print(f"\nNumber of used images: {X_all.shape[0]}")
print(f"\nName and index of classes: {label_map}")

inverse_map = {v: k for k, v in label_map.items()}
distribution = {inverse_map[k]: v for k, v in Counter(y_all).items()}
print(f"Elements per class:  {distribution}")



Total images found: 17092

Number of used images: 17092

Name and index of classes: {'basophil': 0, 'eosinophil': 1, 'erythroblast': 2, 'ig': 3, 'lymphocyte': 4, 'monocyte': 5, 'neutrophil': 6, 'platelet': 7}
Elements per class:  {'basophil': 1218, 'eosinophil': 3117, 'erythroblast': 1551, 'ig': 2895, 'lymphocyte': 1214, 'monocyte': 1420, 'neutrophil': 3329, 'platelet': 2348}


In [47]:
# SPLIT

print("\nSplit labeled / unlabeled / test")

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_all, y_all,
    test_size=TEST_RATIO,
    stratify=y_all,
    random_state=RANDOM_STATE
)

labeled_size = LABELED_RATIO / (LABELED_RATIO + UNLABELED_RATIO)

X_labeled, X_unlabeled, y_labeled, _ = train_test_split(
    X_trainval, y_trainval,
    train_size=labeled_size,
    stratify=y_trainval,
    random_state=RANDOM_STATE
)

print(f"Labeled:   {X_labeled.shape[0]:>6} ({LABELED_RATIO*100:.0f}%)")
print(f"Unlabeled: {X_unlabeled.shape[0]:>6}  ({UNLABELED_RATIO*100:.0f}%)")
print(f"Test:      {X_test.shape[0]:>6} ({TEST_RATIO*100:.0f}%)")



Split labeled / unlabeled / test
Labeled:     3418 (20%)
Unlabeled:  11964  (70%)
Test:        1710 (10%)


In [48]:
# STRANDARDIZATION ON LABELED

scaler = StandardScaler()
scaler.fit(X_labeled)

X_labeled_s   = scaler.transform(X_labeled)
X_unlabeled_s = scaler.transform(X_unlabeled)
X_test_s      = scaler.transform(X_test)


In [49]:
# PCA ON LABELED


pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
pca.fit(X_labeled_s)

X_labeled_pca   = pca.transform(X_labeled_s)
X_unlabeled_pca = pca.transform(X_unlabeled_s)
X_test_pca      = pca.transform(X_test_s)

variance = np.sum(pca.explained_variance_ratio_) * 100
print(f"\nPCA components: {N_COMPONENTS}")
print(f"Variance (useful data info kept after): {variance:.1f}%")

#with 50 components, we had 55% variance
#with 100, we have 68.8%
#with 150, we have 76% ----> i used this also to simplify the upload of .npz
#with 200, we have 81,6%




PCA components: 150
Variance (useful data info kept after): 76.0%


In [50]:
#FINAL FIT


""" A_train = X_labeled_pca
b_train = y_labeled
A_test  = X_test_pca
b_test  = y_test
k       = len(label_map)

print(f"A_train:     {A_train.shape}   for GD / BCGD")
print(f"b_train:     {b_train.shape}")
print(f"A_test:      {A_test.shape}    for final test")
print(f"b_test:      {b_test.shape}")
print(f"X_unlabeled: {X_unlabeled_pca.shape}") """ #just for personal test and report, older version

A_train:     (3418, 150)   for GD / BCGD
b_train:     (3418,)
A_test:      (1710, 150)    for final test
b_test:      (1710,)
X_unlabeled: (11964, 150)


In [51]:
# Calcolo della matrice di covarianza sulle 150 componenti PCA
cov_matrix = np.cov(X_labeled_pca.T)

# Verifica la dimensione: deve essere (150, 150)
print(f"Dimensioni matrice di covarianza: {cov_matrix.shape}")

Dimensioni matrice di covarianza: (150, 150)


In [52]:
# FIRST 100 PIXELS
subset_pixels = X_all[:, 0:100]
cov_raw = np.cov(subset_pixels.T)

# CORRELATION BETWEEN THESE
corr_matrix = np.corrcoef(subset_pixels.T)
avg_corr = np.mean(np.abs(corr_matrix[np.triu_indices(100, k=1)]))
print(f"Average pixel correlation is: {avg_corr:.3f}")



Average pixel correlation is: 0.157


In [53]:
# CENTROID (AVG DIST) FOR EACH CLASS. X_ALL WHOLE DATASET AND 0-7 CLASSES. STILL ONLY LABELED
n_classes = len(np.unique(y_all))
centroids = np.array([X_all[y_all == i].mean(axis=0) for i in range(n_classes)])
dist_matrix = squareform(pdist(centroids)) #eucl dist matrix

#total avg dist
mean_dist = np.mean(dist_matrix[np.triu_indices(n_classes, k=1)])
print(f"Avg dist between classes: {mean_dist:.2f}")

Avg dist between classes: 4.20


In [54]:
# REJOIN DATASETS
X_massive = np.vstack([X_labeled, X_unlabeled])

print(f"Elements and dimension: {X_massive.shape}")


Elements and dimension: (15382, 4096)


In [69]:
# STANDARDIZATION ON LAB+UNLAB
scaler = StandardScaler()
X_massive_scaled = scaler.fit_transform(X_massive)

# PCA ON LAB+UNLAB
pca = PCA(n_components=150)
pca.fit(X_massive_scaled)

explained_variance = np.sum(pca.explained_variance_ratio_) * 100
print(f"Useful infor kept after: {explained_variance:.2f}%")


Useful infor kept after: 74.12%


In [70]:
# LABELED WITH NEW PCA
X_labeled_pca = pca.transform(scaler.transform(X_labeled))

print(f"Labeled elements and dimension: {X_labeled_pca.shape}") 


Labeled elements and dimension: (3418, 150)


In [71]:

X_massive_pca = pca.transform(X_massive_scaled)

# NEW COVARIANCE ON 15000 ELEMENTS
corr_matrix_pca = np.corrcoef(X_massive_pca.T)
upper_tri_pca = corr_matrix_pca[np.triu_indices(corr_matrix_pca.shape[0], k=1)]
avg_corr_pca = np.mean(np.abs(upper_tri_pca))

# CENTROID ON LABELED
new_centroids = np.array([X_labeled_pca[y_labeled == i].mean(axis=0) for i in range(8)])
new_dist_matrix = squareform(pdist(new_centroids))
new_mean_dist = np.mean(new_dist_matrix[np.triu_indices(8, k=1)])


print(f"Avg. correlation: {avg_corr_pca:.6f}")
print(f"Avg distance centroids: {new_mean_dist:.2f} (Compared to 4.20 before)")

Avg. correlation: 0.000037
Avg distance centroids: 21.31 (Compared to 4.20 before)


In [72]:
# KERNEL TRY

from sklearn.decomposition import KernelPCA

#RBF
kpca = KernelPCA(n_components=150, kernel='rbf', gamma=None, n_jobs=-1)
X_labeled_kpca = kpca.fit_transform(scaler.transform(X_labeled))

# New centroids
k_centroids = np.array([X_labeled_kpca[y_labeled == i].mean(axis=0) for i in range(8)])
k_mean_dist = np.mean(pdist(k_centroids))

print(f"Linear PCA: 21.31")
print(f"Kernel PCA: {k_mean_dist:.2f}")

Linear PCA: 21.31
Kernel PCA: 0.20


In [68]:
# PCA FINAL WITH DATA
pca = PCA(n_components=150, random_state=42)
pca.fit(X_massive_scaled)

X_labeled_pca   = pca.transform(scaler.transform(X_labeled))
X_unlabeled_pca = pca.transform(scaler.transform(X_unlabeled))
X_test_pca      = pca.transform(scaler.transform(X_test))

# FINAL FIT
A_train = X_labeled_pca
b_train = y_labeled
A_test  = X_test_pca
b_test  = y_test
k       = len(label_map)


print(f"A_train:     {A_train.shape}   for GD / BCGD")
print(f"b_train:     {b_train.shape}")
print(f"A_test:      {A_test.shape}    for final test")
print(f"b_test:      {b_test.shape}")
print(f"X_unlabeled: {X_unlabeled_pca.shape}")


variance_150 = np.sum(pca.explained_variance_ratio_) * 100
print(f"Final Variance: {variance_150:.2f}%")

A_train:     (3418, 150)   for GD / BCGD
b_train:     (3418,)
A_test:      (1710, 150)    for final test
b_test:      (1710,)
X_unlabeled: (11964, 150)
Final Variance: 74.12%
